# 04 — Position Fundamentals (Value-Investor-Lens)

**Was Du hier siehst:** Für jede Holding-Position die aktuellen Value-Kennzahlen aus `ref_fundamentals_latest`, plus Conviction-Score (gleiche Formel wie im CSP-Screener), plus Bewertungs-Anker via Graham-Number.

**Datenquellen:**
- `pos_holdings` (aktuelle Positionen)
- `ref_fundamentals_latest` (yfinance-Snapshot, weekly refresh über lab_fundamentals)
- `ref_instruments` (für sector, industry, name)
- `mkt_quotes_daily` (Spot für Graham-Number-Vergleich)

**Fragen die Du beantwortest:**
- Welche meiner Positionen sind heute *value-konform* (hohe Conviction)?
- Wo sitze ich in Quality (ROE) vs. Solidity (D/E)?
- Welche Positions sind teurer als Graham es erlauben würde?
- Pro Position: alle 30 Kennzahlen auf einen Blick mit Color-Coding.

**Limitations:**
- Reine Latest-Snapshot-Sicht. Echte 5y-Trends im Margin/Debt-Verlauf werden erst möglich wenn `ref_fundamentals_snapshot` über Monate akkumuliert hat.
- Graham-Number ist Heuristik, nicht Substitute für DCF.
- yfinance liefert für DACH-Mid-Caps oft nur Teilmengen — fehlende Felder werden als `—` dargestellt.


In [ ]:
import sys, pathlib
REPO = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(REPO))

import pandas as pd
import numpy as np
from datetime import date

from modules.analysis._db import session, df
from modules.analysis._plot import setup, COLORS, hline, plt
from modules.screener_csp.conviction import conviction_score, format_conviction_notes
setup()

In [ ]:
# Drill-Down auf eine spezifische Position — None = der erste der niedrigsten Conviction
DRILL_REF_ID = None
GRAHAM_MAX_RATIO = 1.0    # Spot/Graham > 1 = teurer als Graham erlaubt

## 1. Holdings + Fundamentals laden

In [ ]:
with session() as con:
    # Holdings + ref_instruments (für sector etc.)
    holdings = df(con, '''
        SELECT h.ref_instrument_id,
               SUM(h.quantity) AS quantity,
               AVG(h.cost_per_share) AS avg_cost,
               i.symbol, i.name, i.currency, i.asset_type, i.exchange
        FROM pos_holdings h
        LEFT JOIN ref_instruments i USING (ref_instrument_id)
        GROUP BY h.ref_instrument_id, i.symbol, i.name, i.currency, i.asset_type, i.exchange
        ORDER BY i.symbol
    ''')
    # Fundamentals (latest snapshot)
    try:
        fund = df(con, '''
            SELECT *
            FROM ref_fundamentals_latest
            WHERE ref_instrument_id IN (SELECT DISTINCT ref_instrument_id FROM pos_holdings)
        ''')
    except Exception as e:
        print(f'WARN: ref_fundamentals_latest nicht verfügbar ({e}) — Notebook stops here.')
        fund = pd.DataFrame()
    # Latest spot pro instrument (für Graham-Number Vergleich)
    spots = df(con, '''
        WITH ranked AS (
            SELECT ref_instrument_id, ts, close, source,
                   ROW_NUMBER() OVER (
                       PARTITION BY ref_instrument_id
                       ORDER BY ts DESC,
                                CASE source WHEN 'ib' THEN 1 WHEN 'yfinance' THEN 2 ELSE 9 END
                   ) AS rk
            FROM mkt_quotes_daily
            WHERE ref_instrument_id IN (SELECT DISTINCT ref_instrument_id FROM pos_holdings)
        )
        SELECT ref_instrument_id, close AS spot, ts AS spot_ts FROM ranked WHERE rk = 1
    ''')

if holdings.empty:
    print('Portfolio leer — Notebook übersprungen.')
elif fund.empty:
    print('Keine Fundamentals-Daten — bitte lab_fundamentals refresh-all laufen lassen.')
else:
    print(f'Holdings: {len(holdings)}   Fundamentals-Coverage: {len(fund)}/{len(holdings)}')
    # Merge
    pos = holdings.merge(fund, on='ref_instrument_id', how='left', suffixes=('', '_f'))
    pos = pos.merge(spots, on='ref_instrument_id', how='left')
    pos[['symbol', 'name', 'sector', 'industry', 'country', 'market_cap']].head(15)

## 2. Conviction-Score pro Position

Dieselbe Formel wie der CSP-Screener (siehe `modules/screener_csp/conviction.py`). Wenn ein Underlying eine niedrige Conviction hat, ist der CSP-Screener vorsichtiger — und im Reverse: wenn Du es bereits hältst, ist das ein Anlass zur Überprüfung.

In [ ]:
if not holdings.empty and not fund.empty:
    # Berechne Conviction für jede Holding
    def row_conviction(row):
        if pd.isna(row.get('roe')) and pd.isna(row.get('debt_to_equity')) \
           and pd.isna(row.get('fcf_yield')) and pd.isna(row.get('revenue_cagr_5y')) \
           and pd.isna(row.get('operating_margin')):
            return None
        d = row.to_dict()
        return conviction_score(d)
    pos['conv_result'] = pos.apply(row_conviction, axis=1)
    pos['conviction'] = pos['conv_result'].apply(lambda r: r.score if r is not None else np.nan)
    pos['conv_notes'] = pos['conv_result'].apply(
        lambda r: format_conviction_notes(r) if r is not None else 'n/a')

    rank = pos[['symbol', 'sector', 'conviction', 'conv_notes']].dropna(subset=['conviction'])\
             .sort_values('conviction', ascending=False)
    print('Conviction-Ranking:')
    rank

In [ ]:
if not holdings.empty and not fund.empty:
    p = pos.dropna(subset=['conviction']).sort_values('conviction', ascending=True)
    if p.empty:
        print('Keine berechenbare Conviction — alle Fundamentals lückenhaft.')
    else:
        fig, ax = plt.subplots(figsize=(10, max(3, 0.5 * len(p))))
        colors = [COLORS['positive'] if v >= 0.6
                  else (COLORS['warn'] if v >= 0.4 else COLORS['negative'])
                  for v in p['conviction']]
        ax.barh(p['symbol'], p['conviction'], color=colors)
        hline(ax, 0.5, label='neutral', color=COLORS['neutral'])
        hline(ax, 0.7, label='strong',  color=COLORS['positive'])
        ax.set_xlim(0, 1.0)
        ax.set_xlabel('Conviction-Score')
        ax.set_title('Holdings: Value-Conviction (Buffett-Lens)')
        plt.tight_layout(); plt.show()

## 3. Quality- & Valuation-Quadranten

Wo Du sitzt im Mental-Model-Raum:
- **Quality-Quadrant**: ROE (x) vs. Debt/Equity (y, invertiert). Top-rechts = high quality / low leverage.
- **Valuation/Growth**: PE-TTM (x) vs. Revenue-CAGR-5y (y). Top-links = günstig & wachsend (Value-Growth-Sweet-Spot).

In [ ]:
if not holdings.empty and not fund.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

    # Quality Quadrant: ROE x D/E (D/E auf y-Achse invertiert via ylim)
    q1 = pos.dropna(subset=['roe', 'debt_to_equity']).copy()
    if not q1.empty:
        ax1.scatter(q1['roe'] * 100, q1['debt_to_equity'], s=80,
                    c=q1['conviction'].fillna(0.5), cmap='RdYlGn', vmin=0, vmax=1,
                    edgecolors='black', linewidths=0.5)
        for _, r in q1.iterrows():
            ax1.annotate(r['symbol'], (r['roe']*100, r['debt_to_equity']),
                         xytext=(5, 3), textcoords='offset points', fontsize=8)
        hline(ax1, 1.0, label='D/E=1', color=COLORS['warn'])
        ax1.axvline(15, linestyle=':', color=COLORS['ok'], alpha=0.5)
        ax1.set_xlabel('ROE (%)'); ax1.set_ylabel('Debt/Equity')
        ax1.invert_yaxis()  # niedriger D/E = oben
        ax1.set_title('Quality-Quadrant (oben rechts = top)')
    else:
        ax1.text(0.5, 0.5, 'Keine ROE+D/E-Daten', ha='center', va='center', transform=ax1.transAxes)

    # Valuation/Growth Quadrant
    q2 = pos.dropna(subset=['pe_ttm', 'revenue_cagr_5y']).copy()
    if not q2.empty:
        q2 = q2[q2['pe_ttm'] < 80]   # clip Outliers
        ax2.scatter(q2['pe_ttm'], q2['revenue_cagr_5y'] * 100, s=80,
                    c=q2['conviction'].fillna(0.5), cmap='RdYlGn', vmin=0, vmax=1,
                    edgecolors='black', linewidths=0.5)
        for _, r in q2.iterrows():
            ax2.annotate(r['symbol'], (r['pe_ttm'], r['revenue_cagr_5y']*100),
                         xytext=(5, 3), textcoords='offset points', fontsize=8)
        ax2.axvline(20, linestyle=':', color=COLORS['ok'], alpha=0.5)
        hline(ax2, 5, label='5% growth', color=COLORS['ok'])
        ax2.set_xlabel('PE-TTM'); ax2.set_ylabel('Revenue CAGR 5y (%)')
        ax2.set_title('Valuation × Growth (oben links = günstig+wachsend)')
    else:
        ax2.text(0.5, 0.5, 'Keine PE+CAGR-Daten', ha='center', va='center', transform=ax2.transAxes)

    plt.tight_layout(); plt.show()

## 4. Graham-Number — Bewertungs-Sanity-Check

Klassische Benjamin-Graham-Heuristik für *Defensive Investor Stocks*:

$$\text{Graham} = \sqrt{22.5 \times \text{EPS} \times \text{BVPS}}$$

Da wir EPS/BVPS nicht direkt haben, leiten wir sie aus den Ratios ab:
$\text{EPS} = \text{Spot}/\text{PE}$, $\text{BVPS} = \text{Spot}/\text{PB}$.

$$\text{Graham} = \text{Spot} \cdot \sqrt{\tfrac{22.5}{\text{PE} \cdot \text{PB}}}$$

Ratio `Spot/Graham > 1` heißt: aktueller Preis liegt über Grahams Fair-Value-Anker — **kein Buy gemäß Graham**.

**Caveat:** Graham erlaubt das nur für Stocks mit *positiven EPS+BVPS, vernünftiger Solidity, etablierter Historie*. Wachstumstitel und Verlust-Unternehmen sind außerhalb der Formel. Die Zahl ist eine *Stop-Sign-Heuristik*, kein DCF.

In [ ]:
if not holdings.empty and not fund.empty:
    g = pos.dropna(subset=['pe_ttm', 'pb', 'spot']).copy()
    g = g[(g['pe_ttm'] > 0) & (g['pb'] > 0) & (g['spot'] > 0)]
    if g.empty:
        print('Keine Position erfüllt PE>0, PB>0 für Graham-Number-Berechnung.')
    else:
        g['graham_number'] = g['spot'] * np.sqrt(22.5 / (g['pe_ttm'] * g['pb']))
        g['spot_over_graham'] = g['spot'] / g['graham_number']
        g['verdict'] = np.where(
            g['spot_over_graham'] <= GRAHAM_MAX_RATIO, 'cheap',
            np.where(g['spot_over_graham'] <= 1.5, 'fair', 'expensive'))
        cols = ['symbol', 'sector', 'spot', 'pe_ttm', 'pb', 'graham_number',
                'spot_over_graham', 'verdict']
        print(f'Graham-Bewertung (verdict cheap/fair/expensive, threshold={GRAHAM_MAX_RATIO}):')
        g[cols].sort_values('spot_over_graham').to_string()  # noqa
        g[cols].sort_values('spot_over_graham')

In [ ]:
if not holdings.empty and not fund.empty and 'g' in dir() and not g.empty:
    gs = g.sort_values('spot_over_graham', ascending=True)
    fig, ax = plt.subplots(figsize=(10, max(3, 0.5 * len(gs))))
    colors = [COLORS['positive'] if v <= 1.0
              else (COLORS['warn'] if v <= 1.5 else COLORS['negative'])
              for v in gs['spot_over_graham']]
    ax.barh(gs['symbol'], gs['spot_over_graham'], color=colors)
    ax.axvline(1.0, linestyle='--', color=COLORS['ok'], alpha=0.7, label='Graham = Fair')
    ax.axvline(1.5, linestyle='--', color=COLORS['warn'], alpha=0.5, label='1.5× = grenzwertig')
    ax.set_xlabel('Spot / Graham-Number')
    ax.set_title('Wie weit ist der Spot über/unter Grahams Fair-Value?')
    ax.legend(); plt.tight_layout(); plt.show()

## 5. Sector-/Industry-Allokation

In [ ]:
if not holdings.empty and not fund.empty:
    p = pos.copy()
    p['market_value_local'] = p['quantity'] * p['spot']
    by_sector = (p.groupby('sector', dropna=False)['market_value_local']
                  .agg(['sum', 'count'])
                  .rename(columns={'sum': 'mv_local_sum', 'count': 'positions'})
                  .sort_values('mv_local_sum', ascending=False))
    by_sector['pct'] = 100.0 * by_sector['mv_local_sum'] / by_sector['mv_local_sum'].sum()
    print('Allokation per Sector (MV in lokaler Position-Währung, keine FX-Konsolidierung):')
    by_sector

## 6. Per-Position Detail-Karte

Setze `DRILL_REF_ID` in der Param-Zelle oben (z.B. `'IB:AAPL:USD'`), oder lass es `None` — dann wird die Position mit der niedrigsten Conviction als Default genommen (das ist meistens die, die Du Dir ansehen solltest).

In [ ]:
if not holdings.empty and not fund.empty:
    target = DRILL_REF_ID
    if target is None:
        # niedrigste Conviction = der Sorgenkind-Default
        sub = pos.dropna(subset=['conviction']).sort_values('conviction', ascending=True)
        if not sub.empty:
            target = sub.iloc[0]['ref_instrument_id']
        else:
            target = pos.iloc[0]['ref_instrument_id']
    row = pos[pos['ref_instrument_id'] == target]
    if row.empty:
        print(f'Position {target} nicht im Portfolio.')
    else:
        r = row.iloc[0]
        print(f'==> {r["symbol"]:<8s} {r["name"] or "":<40s} ({r["sector"] or "?"}, {r["industry"] or "?"})')
        print(f'    {target}   ts_fund={r["ts"]}   spot={r["spot"]:.2f} {r["currency"]}')
        print(f'    quantity={r["quantity"]:,.0f}  avg_cost={r["avg_cost"]:.2f}')
        if not pd.isna(r['conviction']):
            print(f'    conviction: {r["conviction"]:.2f}   |   {r["conv_notes"]}')
        print()

In [ ]:
if not holdings.empty and not fund.empty and 'r' in dir():
    # Color-coded one-pager mit Schwellwerten je Metrik.
    def fmt_metric(name, val, fmt, thresholds=None):
        if pd.isna(val) or val is None:
            return name, '—', ''
        # Format
        if fmt == 'pct':
            display = f'{val * 100:.2f}%'
        elif fmt == 'ratio':
            display = f'{val:.2f}'
        elif fmt == 'large':
            display = f'{val:,.0f}'
        else:
            display = str(val)
        # Color-Code via thresholds: (lo_bad, hi_good, reversed)
        color = ''
        if thresholds:
            lo, hi, reverse = thresholds
            v = val
            good = (v >= hi) if not reverse else (v <= lo)
            bad  = (v <= lo) if not reverse else (v >= hi)
            if good:
                color = COLORS['positive']
            elif bad:
                color = COLORS['negative']
        return name, display, color

    sections = [
        ('Identity', [
            ('Market Cap',         r['market_cap'],         'large', None),
            ('Enterprise Value',   r['enterprise_value'],   'large', None),
            ('Employees',          r['employees'],          'large', None),
            ('Country',            r['country'],            None,    None),
        ]),
        ('Valuation', [
            ('P/E TTM',     r['pe_ttm'],      'ratio',  (10, 25, False)),
            ('P/E Forward', r['pe_forward'],  'ratio',  (10, 25, False)),
            ('P/B',         r['pb'],          'ratio',  (1.0, 4.0, False)),
            ('P/S TTM',     r['ps_ttm'],      'ratio',  (1.0, 5.0, False)),
            ('EV/EBITDA',   r['ev_ebitda'],   'ratio',  (8, 18, False)),
            ('PEG',         r['peg_ratio'],   'ratio',  (1.0, 2.0, False)),
        ]),
        ('Quality', [
            ('ROE',              r['roe'],              'pct',   (0.05, 0.20, False)),
            ('ROIC',             r['roic'],             'pct',   (0.05, 0.15, False)),
            ('Gross Margin',     r['gross_margin'],     'pct',   (0.20, 0.50, False)),
            ('Operating Margin', r['operating_margin'], 'pct',   (0.05, 0.20, False)),
            ('Net Margin',       r['net_margin'],       'pct',   (0.05, 0.15, False)),
            ('FCF Margin',       r['fcf_margin'],       'pct',   (0.05, 0.15, False)),
        ]),
        ('Solidity', [
            ('Debt/Equity',         r['debt_to_equity'],     'ratio', (0.5, 2.0, True)),
            ('Net Debt/EBITDA',     r['net_debt_to_ebitda'], 'ratio', (1.0, 3.5, True)),
            ('Current Ratio',       r['current_ratio'],      'ratio', (1.0, 2.0, False)),
            ('Interest Coverage',   r['interest_coverage'],  'ratio', (3.0, 10.0, False)),
        ]),
        ('Cashflow & Dividend', [
            ('FCF Yield',     r['fcf_yield'],     'pct', (0.02, 0.07, False)),
            ('Dividend Yield', r['dividend_yield'], 'pct', (0.01, 0.05, False)),
            ('Payout Ratio',  r['payout_ratio'],  'pct', (0.25, 0.70, True)),
        ]),
        ('Growth (5y CAGR)', [
            ('Revenue CAGR',  r['revenue_cagr_5y'],  'pct', (0.02, 0.08, False)),
            ('EPS CAGR',      r['eps_cagr_5y'],      'pct', (0.05, 0.10, False)),
            ('FCF CAGR',      r['fcf_cagr_5y'],      'pct', (0.05, 0.10, False)),
            ('Dividend CAGR', r['dividend_cagr_5y'], 'pct', (0.02, 0.07, False)),
        ]),
    ]

    # Ausgabe als Plain-Text mit ANSI-Colors hätten geschmeckt — aber Jupyter
    # rendert HTML netter. Lass es als clean text-Block.
    from IPython.display import HTML, display
    parts = []
    for sect_name, fields in sections:
        parts.append(f'<h4 style="margin:8px 0 4px 0">{sect_name}</h4>')
        parts.append('<table style="font-family:monospace;font-size:13px;border-collapse:collapse">')
        for name, val, fmt, thr in fields:
            label, display_v, color = fmt_metric(name, val, fmt, thr)
            style = f'color:{color};font-weight:bold' if color else ''
            parts.append(f'<tr><td style="padding:1px 12px 1px 0">{label}</td>'
                          f'<td style="{style}">{display_v}</td></tr>')
        parts.append('</table>')
    display(HTML(''.join(parts)))